# OMIn : 7Baselines, Evaluation and Comparison
## Process Documentation (Full Pipeline with API calls)

**Purpose:** This notebook documents the complete evaluation pipeline including all baseline system implementations, benchmark construction, LLM-as-judge evaluation, and hard unanswerable question testing.

**Systems implemented:**
1. Vanilla LLM — direct Gemini generation, no retrieval
2. Naive RAG — TF-IDF text chunk retrieval
3. KEO-style KG-RAG — flat edge weights, frequency-sorted context (reproduces KEO failure mode)
4. Self-RAG style — LLM self-critique inspired by Asai et al. (2023)
5. BYOKG-RAG simplified — multi-strategy graph retrieval (Mavromatis et al. 2025)
6. Semantic KG-RAG — cosine semantic weighting, no guardrails (our baseline, from notebook 4)
7. Full System — semantic weighting + both guardrails (our contribution, from notebook 5)

**Evaluation pipeline:**
- Run all systems on 65-question expanded benchmark
- Hard unanswerable benchmark (15 questions, 4 types)
- LLM-as-judge scoring (1-5) on answerable questions
- Statistical significance testing
- Threshold ablation analysis

## Setup

In [ ]:
!pip install -q google-genai networkx numpy scikit-learn tqdm rouge-score

In [ ]:
import json, os, time, re
import numpy as np
import networkx as nx
import difflib
from collections import defaultdict, Counter
from tqdm.notebook import tqdm
from google import genai
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

GEMINI_API_KEY   = ""
client           = genai.Client(api_key=GEMINI_API_KEY)
GENERATION_MODEL = "gemini-2.5-flash"
EMBEDDING_MODEL  = "gemini-embedding-001"

# Load all artifacts produced by notebooks 1-5
with open("./output/knowledge_graph.json") as f:
    graph_json = json.load(f)
with open("./output/canonical_embeddings.json") as f:
    entity_embeddings = json.load(f)
with open("./output/relation_embeddings.json") as f:
    relation_embeddings = json.load(f)
with open("./output/records.json") as f:
    records = json.load(f)
with open("./output/graph_schema.json") as f:
    graph_schema = json.load(f)
with open("./output/qa_benchmark_expanded.json") as f:
    benchmark = json.load(f)
with open("./output/eval_results_guarded.json") as f:
    guarded_results = json.load(f)
with open("./output/eval_results_baseline.json") as f:
    semantic_baseline_results = json.load(f)

record_lookup = {r["record_id"]: r for r in records}

# Rebuild NetworkX graph
G = nx.MultiDiGraph()
for node in graph_json["nodes"]:
    G.add_node(node["id"], record_ids=node["record_ids"], frequency=node["frequency"])
for edge in graph_json["edges"]:
    G.add_edge(edge["head"], edge["tail"],
               relation=edge["relation"], weight=edge["weight"],
               record_ids=edge["record_ids"], support_count=edge["support_count"])

# Entity embedding matrix for fast similarity search
entity_list        = list(entity_embeddings.keys())
entity_matrix      = np.array([entity_embeddings[e] for e in entity_list], dtype=np.float32)
norms              = np.linalg.norm(entity_matrix, axis=1, keepdims=True)
entity_matrix_norm = entity_matrix / (norms + 1e-9)
entity_to_idx      = {e: i for i, e in enumerate(entity_list)}

RELATION_TYPES = [
    "CAUSES", "RESULTS_IN", "PART_OF", "LOCATED_IN",
    "REQUIRES_ACTION", "INSPECTED_BY", "REPLACED_BY",
    "PRECEDED_BY", "AFFECTS", "INSTANCE_OF"
]

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Records: {len(records)} | Benchmark: {sum(len(v) for v in benchmark.values())} questions")

## 2. Shared utility functions (used by all systems)

In [ ]:
def safe_parse_json(raw):
    """Robustly parse potentially malformed JSON from LLM responses."""
    raw = raw.strip().replace("```json","").replace("```","").strip()
    for strategy in [
        lambda x: json.loads(x),
        lambda x: json.loads(re.search(r'\{.*\}', x, re.DOTALL).group()),
        lambda x: json.loads(re.sub(r',(\s*[}\]])', r'\1', x))
    ]:
        try:
            return strategy(raw)
        except:
            pass
    return None

def gemini_call_with_retry(client, prompt, model=GENERATION_MODEL, temp=0.2, max_retries=3):
    """Call Gemini with exponential backoff on 503 errors."""
    for attempt in range(max_retries):
        try:
            r = client.models.generate_content(
                model=model, contents=prompt, config={"temperature": temp})
            return r.text.strip()
        except Exception as e:
            if "503" in str(e):
                wait = 30 * (attempt + 1)
                print(f"  503, waiting {wait}s (attempt {attempt+1}/{max_retries})")
                time.sleep(wait)
            else:
                return None
    return None

def embed_query(query, client, model=EMBEDDING_MODEL):
    """Embed a query string using Gemini embedding model."""
    result = client.models.embed_content(model=model, contents=[query])
    emb = np.array(result.embeddings[0].values, dtype=np.float32)
    return emb / (np.linalg.norm(emb) + 1e-9)

def check_domain(query, client):
    """Classify query as IN_DOMAIN (aviation maintenance) or OUT_OF_DOMAIN."""
    prompt = (
        'Classify as IN_DOMAIN (aviation maintenance) or OUT_OF_DOMAIN.\n'
        'Query: "{query}"\n'
        'JSON only: {{"domain": "IN_DOMAIN" or "OUT_OF_DOMAIN", '
        '"answerable": "LIKELY" or "UNLIKELY", "reason": "one sentence"}}'
    ).format(query=query)
    raw = gemini_call_with_retry(client, prompt, temp=0.0)
    result = safe_parse_json(raw) if raw else None
    return result or {"domain": "IN_DOMAIN", "answerable": "LIKELY", "reason": "filter failed"}

print("Utility functions ready.")

## 3. Baseline 1: Vanilla LLM
Direct generation from Gemini using only parametric knowledge. No retrieval, no graph. Serves as the no-retrieval lower bound.

In [ ]:
VANILLA_PROMPT = """You are an aviation maintenance expert.
Answer the following question about aviation maintenance based on your knowledge.
If you don't know, say so explicitly.

Question: {query}
Answer:"""

def vanilla_llm_query(query, client):
    start  = time.time()
    domain = check_domain(query, client)
    if domain["domain"] == "OUT_OF_DOMAIN":
        return {"answer": f"Out of domain: {domain['reason']}",
                "abstained": True, "abstain_reason": "out_of_domain",
                "domain_detected": "OUT_OF_DOMAIN",
                "latency": time.time()-start, "records_retrieved": 0}
    prompt = VANILLA_PROMPT.format(query=query)
    answer = gemini_call_with_retry(client, prompt) or "Error generating answer."
    abstained = any(phrase in answer.lower() for phrase in
                    ["i don't know", "i cannot", "not enough information",
                     "i'm not sure", "cannot provide"])
    return {"answer": answer, "abstained": abstained,
            "abstain_reason": "self_reported" if abstained else None,
            "domain_detected": "IN_DOMAIN",
            "latency": time.time()-start, "records_retrieved": 0}

## 4. Baseline 2: Naive RAG (TF-IDF text chunk retrieval)
Retrieves top-k records by TF-IDF cosine similarity. Standard text-chunk RAG baseline no graph structure.

In [ ]:
# Build TF-IDF index over all 8917 records
print("Building TF-IDF index...")
record_texts = [r["text_original"] for r in records]
tfidf        = TfidfVectorizer(max_features=10000, stop_words="english", ngram_range=(1,2))
tfidf_matrix = tfidf.fit_transform(record_texts)
print(f"TF-IDF index: {tfidf_matrix.shape}")

NAIVE_RAG_PROMPT = """You are an aviation maintenance expert assistant.
Answer the question based ONLY on the provided records.
If records don't contain sufficient information, say so.

Question: {query}

Retrieved Records:
{context}

Answer:"""

def naive_rag_query(query, client, tfidf, tfidf_matrix, records, top_k=15):
    start  = time.time()
    domain = check_domain(query, client)
    if domain["domain"] == "OUT_OF_DOMAIN":
        return {"answer": f"Out of domain: {domain['reason']}",
                "abstained": True, "domain_detected": "OUT_OF_DOMAIN",
                "latency": time.time()-start, "records_retrieved": 0}
    query_vec = tfidf.transform([query])
    sims      = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_idx   = np.argsort(sims)[::-1][:top_k]
    retrieved = [f"[{records[i]['record_id']} | {records[i]['source_file']}]\n{records[i]['text_original']}"
                 for i in top_idx if sims[i] > 0]
    if not retrieved:
        return {"answer": "No relevant records found.", "abstained": True,
                "domain_detected": "IN_DOMAIN", "latency": time.time()-start, "records_retrieved": 0}
    context = "\n\n".join(retrieved)
    prompt  = NAIVE_RAG_PROMPT.format(query=query, context=context[:4000])
    answer  = gemini_call_with_retry(client, prompt) or "Error."
    abstained = any(p in answer.lower() for p in
                    ["do not contain", "not enough information", "cannot answer",
                     "insufficient", "no information"])
    return {"answer": answer, "abstained": abstained, "domain_detected": "IN_DOMAIN",
            "latency": time.time()-start, "records_retrieved": len(retrieved)}

## 5. Baseline 3: KEO-style KG-RAG
Reproduces the KEO approach: top-k seed nodes by cosine similarity (no MMR diversity), 2-hop expansion, all edges returned with flat weight=1, sorted by frequency (support_count) as KEO does. This directly validates our critique of KEO's flat-weight approach.

In [ ]:
def find_seed_nodes_flat(query_emb, entity_matrix_norm, entity_list, G, top_k=10):
    """Top-k by cosine similarity — no MMR, reproduces KEO's entity retrieval."""
    sims    = entity_matrix_norm @ query_emb
    top_idx = np.argsort(sims)[::-1]
    seeds   = []
    for idx in top_idx:
        entity = entity_list[idx]
        if entity in G.nodes:
            seeds.append({"node": entity, "similarity": float(sims[idx])})
        if len(seeds) >= top_k:
            break
    return seeds

def expand_subgraph_flat(seed_nodes, G, hops=2):
    """2-hop expansion returning ALL edges with flat weight=1 (KEO-style)."""
    visited  = set(s["node"] for s in seed_nodes)
    frontier = set(visited)
    for _ in range(hops):
        new_frontier = set()
        for node in frontier:
            if node not in G: continue
            for nb in list(G.successors(node)) + list(G.predecessors(node)):
                if nb not in visited:
                    new_frontier.add(nb)
                    visited.add(nb)
        frontier = new_frontier
    edges = []
    for u, v, data in G.edges(data=True):
        if u in visited and v in visited:
            edges.append({"head": u, "tail": v, "relation": data["relation"],
                          "record_ids": data["record_ids"],
                          "support_count": data["support_count"], "relevance_score": 1.0})
    return edges, visited

KEO_PROMPT = """You are an aviation maintenance expert.
Answer the question using the knowledge graph context below.
If insufficient, say so.

Question: {query}

Knowledge Graph Context:
{kg_context}

Answer:"""

def keo_style_query(query, client, G, entity_matrix_norm, entity_list,
                     entity_embeddings, record_lookup, top_k=10, hops=2):
    start  = time.time()
    domain = check_domain(query, client)
    if domain["domain"] == "OUT_OF_DOMAIN":
        return {"answer": f"Out of domain: {domain['reason']}", "abstained": True,
                "domain_detected": "OUT_OF_DOMAIN", "latency": time.time()-start,
                "records_retrieved": 0, "edges_in_subgraph": 0}
    query_emb    = embed_query(query, client)
    seeds        = find_seed_nodes_flat(query_emb, entity_matrix_norm, entity_list, G, top_k)
    edges, _     = expand_subgraph_flat(seeds, G, hops)
    edges_sorted = sorted(edges, key=lambda x: -x["support_count"])[:50]
    kg_lines     = [f"{e['head']} --[{e['relation']}]--> {e['tail']}" for e in edges_sorted]
    kg_context   = "\n".join(kg_lines)
    record_scores = defaultdict(float)
    for e in edges_sorted:
        for rid in e["record_ids"]:
            record_scores[rid] += e["support_count"]
    top_recs    = sorted(record_scores.items(), key=lambda x: -x[1])[:15]
    rec_context = "".join(f"[{rid}] {record_lookup[rid]['text_original']}\n"
                           for rid, _ in top_recs if rid in record_lookup)
    full_context = f"GRAPH EDGES:\n{kg_context}\n\nSOURCE RECORDS:\n{rec_context}"
    prompt       = KEO_PROMPT.format(query=query, kg_context=full_context[:4000])
    answer       = gemini_call_with_retry(client, prompt) or "Error."
    abstained    = any(p in answer.lower() for p in
                       ["not enough", "cannot answer", "insufficient", "do not contain"])
    return {"answer": answer, "abstained": abstained, "domain_detected": "IN_DOMAIN",
            "latency": time.time()-start, "records_retrieved": len(top_recs),
            "edges_in_subgraph": len(edges)}

## 6. Baseline 4: Self-RAG style
Inspired by Asai et al. (2023). Three-step pipeline: (1) decide if retrieval is needed, (2) retrieve and generate, (3) self-critique to verify support. Note: this is a simplified adaptation true Self-RAG requires fine-tuning special reflection tokens which we do not implement.

In [ ]:
SELFRAG_RETRIEVE_PROMPT = (
    'You are an aviation maintenance expert.\n'
    'First decide: do you need to retrieve information to answer this query?\n'
    'Query: "{query}"\n'
    'JSON only: {{"need_retrieval": true or false, "reason": "why"}}'
)

SELFRAG_CRITIQUE_PROMPT = (
    "Evaluate this answer to the aviation maintenance query.\n"
    "Query: {query}\nRetrieved context: {context}\nAnswer: {answer}\n\n"
    "Rate on:\n"
    "- isrel: is the retrieved context relevant? (true/false)\n"
    "- issup: is the answer supported by the context? (true/false)\n"
    "- isuse: is the answer useful? (1-5)\n\n"
    "JSON only: {{\"isrel\": bool, \"issup\": bool, \"isuse\": 1-5}}"
)

def selfrag_style_query(query, client, tfidf, tfidf_matrix, records, top_k=10):
    start  = time.time()
    domain = check_domain(query, client)
    if domain["domain"] == "OUT_OF_DOMAIN":
        return {"answer": "Out of domain.", "abstained": True,
                "domain_detected": "OUT_OF_DOMAIN", "latency": time.time()-start,
                "records_retrieved": 0, "critique": None}
    # Step 1: decide retrieval
    raw = gemini_call_with_retry(client, SELFRAG_RETRIEVE_PROMPT.format(query=query), temp=0.0)
    retrieve_decision = safe_parse_json(raw) if raw else {"need_retrieval": True}
    context, n_retrieved = "", 0
    if retrieve_decision.get("need_retrieval", True):
        sims    = cosine_similarity(tfidf.transform([query]), tfidf_matrix).flatten()
        top_idx = np.argsort(sims)[::-1][:top_k]
        ctx_parts = [f"[{records[i]['record_id']}] {records[i]['text_original']}"
                     for i in top_idx if sims[i] > 0]
        context, n_retrieved = "\n".join(ctx_parts), len(ctx_parts)
    # Step 2: generate
    gen_prompt = (NAIVE_RAG_PROMPT.format(query=query, context=context[:3000]) if context
                  else VANILLA_PROMPT.format(query=query))
    answer = gemini_call_with_retry(client, gen_prompt) or "Error."
    # Step 3: self-critique
    crit_raw = gemini_call_with_retry(
        client, SELFRAG_CRITIQUE_PROMPT.format(
            query=query, context=context[:1000], answer=answer[:500]), temp=0.0)
    critique = safe_parse_json(crit_raw) if crit_raw else None
    abstained = False
    if critique and not critique.get("issup", True):
        answer    = f"[Self-critique: answer not supported by evidence] {answer}"
        abstained = True
    return {"answer": answer, "abstained": abstained, "domain_detected": "IN_DOMAIN",
            "latency": time.time()-start, "records_retrieved": n_retrieved, "critique": critique}

## 7. Baseline 5: BYOKG-RAG Simplified
Based on Mavromatis et al. (2025) 'BYOKG-RAG: Multi-Strategy Graph Retrieval for Knowledge Graph Question Answering' (Amazon Science).

**Implemented components:**
- KG-Linker: LLM generates entities, relation paths, draft answer entities, and draft answer from a single call
- Multi-strategy entity linking: embedding similarity (top-m) + fuzzy string matching
- Path retrieval: BFS expansion from linked entities + Dijkstra shortest paths to draft answer entities
- Scoring-based triplet retrieval: cosine similarity ranking

**Omitted** (require graph DB infrastructure not available in our setup):
- OpenCypher query generation (requires Neo4j/Neptune)
- Iterative refinement loop (TR=2 iterations)
- Full agentic step-by-step traversal

In [ ]:
BYOKG_LINKER_PROMPT = """You are a knowledge graph expert for aviation maintenance.
Given a question and the graph schema, generate graph artifacts for retrieval.

GRAPH SCHEMA:
- Node types: aviation entities (components, failures, actions, conditions)
- Relation types: CAUSES, RESULTS_IN, PART_OF, LOCATED_IN, REQUIRES_ACTION,
  INSPECTED_BY, REPLACED_BY, PRECEDED_BY, AFFECTS, INSTANCE_OF

GRAPH CONTEXT (if available):
{graph_context}

Question: {question}

Perform these tasks:
1. ENTITY EXTRACTION: Extract all topic entities from the question
2. PATH IDENTIFICATION: Identify relation paths that connect entities to answers
3. DRAFT ANSWER: Generate a draft answer using your knowledge

Respond ONLY with JSON:
{{
  "entities": ["list of extracted entities from question"],
  "relation_paths": [["RELATION1", "RELATION2"], ["RELATION1"]],
  "draft_answer_entities": ["list of likely answer entities"],
  "draft_answer": "brief draft answer"
}}"""

def byokg_entity_linking(generated_entities, G, entity_embeddings,
                          entity_matrix_norm, entity_list, entity_to_idx, top_m=3):
    """Multi-strategy linking: embedding similarity + fuzzy string match."""
    linked = set()
    for gen_entity in generated_entities:
        # Strategy 1: embedding similarity
        try:
            result = client.models.embed_content(model=EMBEDDING_MODEL, contents=[gen_entity])
            emb    = np.array(result.embeddings[0].values, dtype=np.float32)
            emb   /= (np.linalg.norm(emb) + 1e-9)
            sims   = entity_matrix_norm @ emb
            for idx in np.argsort(sims)[::-1][:top_m]:
                node = entity_list[idx]
                if node in G.nodes and sims[idx] > 0.7:
                    linked.add(node)
        except:
            pass
        # Strategy 2: fuzzy string match
        for m in difflib.get_close_matches(gen_entity.lower(), list(G.nodes()), n=top_m, cutoff=0.6):
            linked.add(m)
    return list(linked)

def byokg_path_retrieval(linked_entities, draft_answer_entities, G, max_paths=50):
    """BFS expansion + Dijkstra shortest paths (Follow-Paths + Shortest-Paths)."""
    retrieved_triplets, visited_edges = [], set()
    for entity in linked_entities[:5]:
        if entity not in G: continue
        for u, v, data in list(G.edges(entity, data=True)) + list(G.in_edges(entity, data=True)):
            key = (u, data["relation"], v)
            if key not in visited_edges:
                visited_edges.add(key)
                retrieved_triplets.append({"head": u, "relation": data["relation"], "tail": v,
                    "record_ids": data["record_ids"], "support_count": data["support_count"],
                    "retrieval_method": "bfs_expansion"})
    undirected = G.to_undirected()
    for src in linked_entities[:3]:
        for tgt in [e for e in draft_answer_entities if e in G][:3]:
            if src == tgt or src not in G or tgt not in G: continue
            try:
                for i, j in zip(*[nx.shortest_path(undirected, src, tgt)[:-1],
                                   nx.shortest_path(undirected, src, tgt)[1:]]):
                    for _, _, data in G.edges(i, data=True):
                        key = (i, data["relation"], j)
                        if key not in visited_edges:
                            visited_edges.add(key)
                            retrieved_triplets.append({"head": i, "relation": data["relation"],
                                "tail": j, "record_ids": data["record_ids"],
                                "support_count": data["support_count"],
                                "retrieval_method": "shortest_path"})
            except (nx.NetworkXNoPath, nx.NodeNotFound):
                pass
    return retrieved_triplets[:max_paths]

def byokg_scoring_retrieval(query_emb, triplets, entity_embeddings, relation_embeddings, top_k=30):
    """Score triplets by semantic similarity to query (Section 3.5 of BYOKG-RAG)."""
    scored = []
    for t in triplets:
        h_e = entity_embeddings.get(t["head"])
        t_e = entity_embeddings.get(t["tail"])
        r_e = relation_embeddings.get(t["relation"])
        if h_e is None or t_e is None or r_e is None: continue
        h = np.array(h_e, dtype=np.float32); h /= (np.linalg.norm(h) + 1e-9)
        t = np.array(t_e, dtype=np.float32); t /= (np.linalg.norm(t) + 1e-9)
        r = np.array(r_e, dtype=np.float32); r /= (np.linalg.norm(r) + 1e-9)
        score = 0.4*float(np.dot(query_emb, h)) + 0.4*float(np.dot(query_emb, t)) + \
                0.2*float(np.dot(query_emb, r))
        scored.append({**t, "relevance_score": score})
    scored.sort(key=lambda x: -x["relevance_score"])
    return scored[:top_k]

BYOKG_GENERATION_PROMPT = """You are an aviation maintenance expert.
Answer the question using the retrieved knowledge graph context and source records.
If insufficient, say so explicitly.

Question: {query}
KG-Linker Draft Answer: {draft_answer}
Retrieved Graph Triplets:
{triplet_context}
Source Records:
{record_context}

Final Answer:"""

def byokg_rag_query(query, client, G, entity_matrix_norm, entity_list, entity_to_idx,
                     entity_embeddings, relation_embeddings, record_lookup, graph_schema):
    start  = time.time()
    domain = check_domain(query, client)
    if domain["domain"] == "OUT_OF_DOMAIN":
        return {"answer": f"Out of domain: {domain['reason']}", "abstained": True,
                "domain_detected": "OUT_OF_DOMAIN", "latency": time.time()-start,
                "records_retrieved": 0, "linked_entities": [], "retrieval_methods": []}
    # Step 1: KG-Linker
    raw       = gemini_call_with_retry(client, BYOKG_LINKER_PROMPT.format(question=query, graph_context=""), temp=0.1)
    artifacts = safe_parse_json(raw) if raw else {}
    gen_ents  = artifacts.get("entities", [])
    ans_ents  = artifacts.get("draft_answer_entities", [])
    draft_ans = artifacts.get("draft_answer", "")
    # Step 2: Multi-strategy entity linking
    linked = byokg_entity_linking(gen_ents + ans_ents, G, entity_embeddings,
                                   entity_matrix_norm, entity_list, entity_to_idx)
    if not linked:  # fallback: embed query
        q_emb  = embed_query(query, client)
        top_idx = np.argsort(entity_matrix_norm @ q_emb)[::-1][:5]
        linked  = [entity_list[i] for i in top_idx if entity_list[i] in G.nodes]
    # Step 3: Path retrieval
    path_triplets   = byokg_path_retrieval(linked, ans_ents, G)
    # Step 4: Scoring retrieval
    query_emb       = embed_query(query, client)
    scored_triplets = byokg_scoring_retrieval(query_emb, path_triplets, entity_embeddings, relation_embeddings)
    # Step 5: Record retrieval
    rec_scores = defaultdict(float)
    for t in scored_triplets:
        for rid in t.get("record_ids", []):
            rec_scores[rid] += t["relevance_score"]
    top_recs  = sorted(rec_scores.items(), key=lambda x: -x[1])[:15]
    rec_texts = [f"[{rid} | {record_lookup[rid]['source_file']}] {record_lookup[rid]['text_original']}"
                 for rid, _ in top_recs if rid in record_lookup]
    # Step 6: Generate
    triplet_ctx = "\n".join(f"{t['head']} --[{t['relation']}]--> {t['tail']} "
                             f"(score={t['relevance_score']:.3f}, via {t['retrieval_method']})"
                             for t in scored_triplets[:15])
    prompt = BYOKG_GENERATION_PROMPT.format(query=query, draft_answer=draft_ans,
        triplet_context=triplet_ctx, record_context="\n".join(rec_texts[:10])[:3000])
    answer = gemini_call_with_retry(client, prompt) or "Error."
    abstained = any(p in answer.lower() for p in
                    ["not enough", "cannot answer", "insufficient", "do not contain", "no information"])
    return {"answer": answer, "abstained": abstained, "domain_detected": "IN_DOMAIN",
            "latency": time.time()-start, "records_retrieved": len(top_recs),
            "linked_entities": linked[:5], "draft_answer": draft_ans,
            "retrieval_methods": list(set(t["retrieval_method"] for t in scored_triplets))}

## running all baselines on benchmark (checkpointed)

In [ ]:
BASELINE_CHECKPOINT = "./output/baseline_checkpoint.json"

if os.path.exists(BASELINE_CHECKPOINT):
    with open(BASELINE_CHECKPOINT) as f:
        all_baseline_results = json.load(f)
    print("Resumed from checkpoint:")
    for sys_name, qtypes in all_baseline_results.items():
        print(f"  {sys_name}: {sum(len(v) for v in qtypes.values())} done")
else:
    all_baseline_results = {sys: {qt: [] for qt in benchmark}
                            for sys in ["vanilla_llm", "naive_rag", "keo_style",
                                        "selfrag_style", "byokg_rag"]}

for sys_key in ["byokg_rag"]:
    if sys_key not in all_baseline_results:
        all_baseline_results[sys_key] = {qt: [] for qt in benchmark}

SYSTEMS = {
    "vanilla_llm":   lambda q: vanilla_llm_query(q, client),
    "naive_rag":     lambda q: naive_rag_query(q, client, tfidf, tfidf_matrix, records),
    "keo_style":     lambda q: keo_style_query(q, client, G, entity_matrix_norm,
                                                entity_list, entity_embeddings, record_lookup),
    "selfrag_style": lambda q: selfrag_style_query(q, client, tfidf, tfidf_matrix, records),
    "byokg_rag":     lambda q: byokg_rag_query(q, client, G, entity_matrix_norm,
                                                entity_list, entity_to_idx, entity_embeddings,
                                                relation_embeddings, record_lookup, graph_schema),
}

for sys_name, sys_fn in SYSTEMS.items():
    print(f"\n{'='*50}\nSystem: {sys_name}")
    for qtype, questions in benchmark.items():
        done     = {r["question"] for r in all_baseline_results[sys_name].get(qtype, [])}
        remaining = [q for q in questions
                     if (q["question"] if isinstance(q, dict) else q) not in done]
        if not remaining:
            print(f"  {qtype}: already complete")
            continue
        print(f"  {qtype}: {len(remaining)} remaining")
        for i, q_data in enumerate(tqdm(remaining, desc=f"{sys_name}/{qtype}")):
            query  = q_data["question"] if isinstance(q_data, dict) else q_data
            result = sys_fn(query)
            result["question"] = query
            if qtype == "out_of_domain":
                result["domain_correct"]     = result["domain_detected"] == "OUT_OF_DOMAIN"
                result["abstention_correct"] = result["domain_detected"] == "OUT_OF_DOMAIN"
            elif qtype == "in_domain_unanswerable":
                result["domain_correct"]     = result["domain_detected"] == "IN_DOMAIN"
                result["abstention_correct"] = result.get("abstained", False)
            else:
                result["domain_correct"]     = result["domain_detected"] == "IN_DOMAIN"
                result["abstention_correct"] = not result.get("abstained", False)
            all_baseline_results[sys_name].setdefault(qtype, []).append(result)
            time.sleep(2.0)
            if (i + 1) % 5 == 0:
                with open(BASELINE_CHECKPOINT, "w") as f:
                    json.dump(all_baseline_results, f, indent=2, default=str)
                tqdm.write("  Checkpoint saved")

with open(BASELINE_CHECKPOINT, "w") as f:
    json.dump(all_baseline_results, f, indent=2, default=str)
with open("./output/baseline_results.json", "w") as f:
    json.dump(all_baseline_results, f, indent=2, default=str)

## hard unanswerable benchmark
15 questions grounded in real database identifiers, requiring capabilities beyond graph retrieval: specific record lookup by FAA code, statistical aggregation, and information not present in the database. Designed to stress-test abstention behavior. (previous benchmark for unanswerable was not satisfying)

In [ ]:
HARD_UNANSWERABLE = [
    # Type A: Specific record lookup — requires finding exact FAA incident by code
    {"question": "What specific aircraft registration number was involved in incident 19750315005389A?",
     "expected_behavior": "abstain", "type": "specific_record_lookup"},
    {"question": "What was the exact maintenance action taken for annotation record IDENT 100007?",
     "expected_behavior": "abstain", "type": "specific_record_lookup"},
    {"question": "What happened to aircraft N49815 on October 24 2005?",
     "expected_behavior": "abstain", "type": "specific_record_lookup"},
    {"question": "What type of aircraft was involved in incident code 19760215007629A?",
     "expected_behavior": "abstain", "type": "specific_record_lookup"},
    {"question": "What was the pilot's error in FAA incident 19751209037899A?",
     "expected_behavior": "abstain", "type": "specific_record_lookup"},
    {"question": "What is the exact text of FAA incident report number 19760531024959A?",
     "expected_behavior": "abstain", "type": "specific_record_lookup"},
    # Type B: Aggregation — graph cannot count or perform statistics
    {"question": "How many aviation incidents in this database involved water contamination in fuel systems?",
     "expected_behavior": "abstain", "type": "aggregation"},
    {"question": "What percentage of engine failures in the database occurred during takeoff versus landing?",
     "expected_behavior": "abstain", "type": "aggregation"},
    {"question": "Which year had the highest number of hydraulic system failures in the FAA records?",
     "expected_behavior": "abstain", "type": "aggregation"},
    # Type C: Missing information — not captured in FAA incident narratives
    {"question": "What was the weather condition at the time of incident 19750905025029A?",
     "expected_behavior": "abstain", "type": "missing_information"},
    {"question": "What was the total repair cost for the landing gear collapse described in FAA record 1851?",
     "expected_behavior": "abstain", "type": "missing_information"},
    {"question": "How many flight hours had the engine accumulated before the failure in incident 19760606015529A?",
     "expected_behavior": "abstain", "type": "missing_information"},
    {"question": "What was the name of the mechanic who performed the fuel servo replacement in annotation 100002?",
     "expected_behavior": "abstain", "type": "missing_information"},
    # Type D: Full database scan required — beyond graph retrieval capability
    {"question": "Which specific aircraft tail number appears most frequently across all incident reports?",
     "expected_behavior": "abstain", "type": "full_scan_required"},
    {"question": "Are there any incidents where both a fuel system failure and a landing gear failure occurred in the same event?",
     "expected_behavior": "abstain", "type": "full_scan_required"},
]

print(f"Hard unanswerable questions: {len(HARD_UNANSWERABLE)}")
print("Type distribution:", Counter(q["type"] for q in HARD_UNANSWERABLE))

STRICT_ABSTAIN_PHRASES = [
    "i cannot answer", "i don't have", "cannot be determined",
    "no relevant records", "outside the scope", "i am unable",
    "not available in", "specific record", "cannot find",
    "do not have access", "out of domain"
]

REEVAL_FILE = "./output/hard_unansw_results.json"
hard_results = json.load(open(REEVAL_FILE)) if os.path.exists(REEVAL_FILE) else {}

systems_hard = {
    "vanilla_llm":   lambda q: vanilla_llm_query(q, client),
    "naive_rag":     lambda q: naive_rag_query(q, client, tfidf, tfidf_matrix, records),
    "keo_style":     lambda q: keo_style_query(q, client, G, entity_matrix_norm,
                                                entity_list, entity_embeddings, record_lookup),
    "selfrag_style": lambda q: selfrag_style_query(q, client, tfidf, tfidf_matrix, records),
    "byokg_rag":     lambda q: byokg_rag_query(q, client, G, entity_matrix_norm,
                                                entity_list, entity_to_idx, entity_embeddings,
                                                relation_embeddings, record_lookup, graph_schema),
}

for sys_name, sys_fn in systems_hard.items():
    if sys_name in hard_results:
        print(f"  {sys_name}: already done")
        continue
    print(f"\n {sys_name}")
    hard_results[sys_name] = []
    for q_data in tqdm(HARD_UNANSWERABLE, desc=sys_name):
        result = sys_fn(q_data["question"])
        result["question"]      = q_data["question"]
        result["q_type_detail"] = q_data["type"]
        answer_lower = result.get("answer", "").lower()
        result["strict_abstained"] = (
            any(p in answer_lower for p in STRICT_ABSTAIN_PHRASES) or
            result.get("domain_detected") == "OUT_OF_DOMAIN"
        )
        hard_results[sys_name].append(result)
        time.sleep(2.0)

with open(REEVAL_FILE, "w") as f:
    json.dump(hard_results, f, indent=2, default=str)

print("\n HARD UNANSWERABLE RESULTS")
for sys_name, results in hard_results.items():
    n    = len(results)
    abs_ = sum(1 for r in results if r.get("strict_abstained", False))
    by_t = Counter(r["q_type_detail"] for r in results if r.get("strict_abstained"))
    print(f"  {sys_name:<20} {abs_}/{n}={abs_/n:.1%}  {dict(by_t)}")

## 10. LLM-as-Judge evaluation
Gemini 2.5 Flash scores each answer on 5 criteria (1-5 scale): correctness, completeness, grounding, safety, clarity. Applied to all systems on answerable + multi-hop questions. Checkpointed every 5 answers.

In [ ]:
LLM_JUDGE_PROMPT = """You are an expert aviation maintenance evaluator.
Score the following answer to an aviation maintenance question on a 1-5 scale.

QUESTION: {question}
ANSWER TO EVALUATE: {answer}
REFERENCE CONTEXT (what a good answer should cover):
{reference}

Score on these criteria (1-5 each):
1. CORRECTNESS: Is the answer factually accurate for aviation maintenance?
2. COMPLETENESS: Does it cover the key aspects needed to answer the question?
3. GROUNDING: Is the answer specific and grounded, or vague and generic?
4. SAFETY: Would following this answer maintain aviation safety standards?
5. CLARITY: Is the answer clear and actionable?

Respond ONLY with JSON:
{{"correctness": 1-5, "completeness": 1-5, "grounding": 1-5,
  "safety": 1-5, "clarity": 1-5, "overall": 1-5, "reasoning": "one sentence"}}"""

def llm_judge_answer(question, answer, reference_answers, client):
    reference = "\n".join(reference_answers[:3]) if reference_answers else "No reference."
    prompt    = LLM_JUDGE_PROMPT.format(question=question, answer=answer[:800],
                                         reference=reference[:500])
    for attempt in range(3):
        try:
            r      = client.models.generate_content(model=GENERATION_MODEL, contents=prompt,
                                                     config={"temperature": 0.0})
            result = safe_parse_json(r.text)
            if result: return result
        except Exception as e:
            if "503" in str(e): time.sleep(30 * (attempt + 1))
            else: break
    return {"correctness": 3, "completeness": 3, "grounding": 3,
            "safety": 3, "clarity": 3, "overall": 3, "reasoning": "judge failed"}

# reference: full system's answers (context for judge, not gold standard)
full_sys_answers = {r["question"]: r["answer"]
                    for qt in ["in_domain_answerable", "multi_hop"]
                    for r in guarded_results.get(qt, []) if not r.get("abstained")}

JUDGE_FILE = "./output/llm_judge_results.json"
judge_results = json.load(open(JUDGE_FILE)) if os.path.exists(JUDGE_FILE) else {}

systems_to_judge = {
    "vanilla_llm":   all_baseline_results.get("vanilla_llm", {}),
    "naive_rag":     all_baseline_results.get("naive_rag", {}),
    "keo_style":     all_baseline_results.get("keo_style", {}),
    "selfrag_style": all_baseline_results.get("selfrag_style", {}),
    "byokg_rag":     all_baseline_results.get("byokg_rag", {}),
    "semantic_kgrag": semantic_baseline_results,
    "full_system":   guarded_results,
}

for sys_name, results_by_type in systems_to_judge.items():
    judge_results.setdefault(sys_name, {})
    print(f"\n Judging: {sys_name}")
    for qtype in ["in_domain_answerable", "multi_hop"]:
        results     = results_by_type.get(qtype, [])
        already     = set(judge_results[sys_name].get(qtype, {}).keys())
        judge_results[sys_name].setdefault(qtype, {})
        to_judge    = [r for r in results if r.get("question") not in already
                       and not r.get("abstained", False)]
        if not to_judge:
            print(f"  {qtype}: complete")
            continue
        print(f"  {qtype}: judging {len(to_judge)} answers")
        for i, r in enumerate(tqdm(to_judge, desc=f"{sys_name}/{qtype}")):
            q = r.get("question", "")
            scores = llm_judge_answer(q, r.get("answer", ""), [full_sys_answers.get(q, "")], client)
            judge_results[sys_name][qtype][q] = {"scores": scores, "answer_preview": r.get("answer","")[:200]}
            time.sleep(2.0)
            if (i + 1) % 5 == 0:
                with open(JUDGE_FILE, "w") as f: json.dump(judge_results, f, indent=2)
                tqdm.write("  Judge checkpoint saved")

with open(JUDGE_FILE, "w") as f:
    json.dump(judge_results, f, indent=2)

print("\n LLM-AS-JUDGE RESULTS (1-5 scale)")
print(f"{'System':<22}" + "".join(f"{m[:9]:>10}" for m in
      ["correctness", "completeness", "grounding", "safety", "clarity", "overall"]))
print("-" * 82)
for sys_name in systems_to_judge:
    row = f"{sys_name:<22}"
    for m in ["correctness", "completeness", "grounding", "safety", "clarity", "overall"]:
        vals = [float(d["scores"][m]) for qt in judge_results.get(sys_name, {}).values()
                for d in qt.values() if d.get("scores", {}).get(m) is not None]
        row += f"{np.mean(vals):>10.2f}" if vals else f"{'N/A':>10}"
    print(row)